In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
df_materials = pd.read_parquet('/content/drive/MyDrive/filtered_materials_encoded_time_cut.parquet')
df_bo_me = pd.read_parquet('/content/drive/MyDrive/merged_cleaned_balanced_15k.parquet')

In [ ]:
plt.figure(figsize=(12, 10))
import seaborn as sns

c = df.corr()
sns.heatmap(c, annot=True, cmap="twilight")

In [ ]:
label_col = 'book_state'
y = df_bo_me[label_col]
print(df_bo_me[label_col].value_counts())

In [ ]:
'''datetime_cols = [
created_at', 'book_stamp'

for col in datetime_cols:
if col in df_bo_me.columns:
df_bo_me[f'{col}_hour'] = df_bo_me[col].dt.hour
df_bo_me[f'{col}_weekday'] = df_bo_me[col].dt.weekday
df_bo_me[f'{col}_time_since_start' ] = (df_bo_me[col] - df_bo_me[col].min()).dt.total_seconds()
df_bo_me.drop(columns=col, inplace=True)

numerical_cols = [
'measure_step_number', 'measure_value', 'measurement_name_encoded', 'measurement_unit_encoded', 'is_within_limits', 'workstep_number_mes'

1

]

categorical_cols = [
'serial_number_id', 'station_id', 'booking_id', 'part_group'

df_encoded = pd.get_dummies(df_bo_me[categorical_cols], drop_first=True)

df_bo_me[numerical_cols] = df_bo_me[numerical_cols].fillna(df_bo_me[numerical_cols].mean())

X = pd.concat([df_bo_me[numerical_cols], df_encoded], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, stratify=y, random_state=42

)'''

In [ ]:
'''#Dummy variable for ‘Geography’ column
geography = pd.get_dummies(X[‘Geography’], drop_first = True)
#Dummy variable for ‘Gender’ column
gender = pd.get_dummies(X[‘Gender’], drop_first = True)
#Dropping the original ‘Geography’ and ‘Gender’ columns
X = X.drop([‘Geography’,’Gender’], axis = 1)
#Adding the dummy columns to the dataset
X = pd.concat([X,geography,gender], axis = 1)
X.head()'''

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost

classifier = xgboost.XGBClassifier()

In [ ]:
params = {
“learning_rate”: [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
“max_depth”: [3, 4, 5, 6, 8, 10, 12, 15],
“min_child_weight”: [1, 3, 5, 7],
“gamma”: [0.0, 0.1, 0.2, 0.3, 0.4],
“colsample_bytree”: [0.3, 0.4, 0.5, 0.7]
}

In [ ]:
rs_model = RandomizedSearchCV(classifier, param_distributions=params, n_iter=5, scoring='roc_auc', n_jobs=-1, cv=5,
                              verbose=3)

In [ ]:
rs_model.fit(X, y)

In [ ]:
rs_model.best_estimator_

In [ ]:
classifier = xgboost.XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1, colsample_bynode=1,
                                   colsample_bytree=0.7, gamma=0.3, learning_rate=0.05, max_delta_step=0, max_depth=6,
                                   min_child_weight=1, missing=None, n_estimators=100, n_jobs=1, nthread=None,
                                   objective='binary:logistic', random_state=0, reg_alpha=0, reg_lambda=1,
                                   scale_pos_weight=1, seed=None, silent=None, subsample=1, verbosity=1)

In [ ]:
from sklearn.model_selection import cross_val_score

score = cross_val_score(classifier, X, y, cv=10)
print(score)

In [ ]:
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Confusion Matrix berechnen
cm = confusion_matrix(y_test, y_pred)
labels = ['no error (0)', 'error (1)']

# Als Heatmap plotten
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix – Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Random Forest')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.nlargest(8).plot(kind='barh')
plt.title("Top 8 Feature Importances of Random Forest")
plt.tight_layout()
plt.show()